In [ ]:
'''
本实验用到的库有：
numpy、matplotlib、scipy
scikit-learn、pandas、seaborn
'''

## Task1（30%）
- 使用 Pandas 对数据进行初步处理

#### Q1.(4%)
 -    读取数据


In [1]:
import pandas as pd
df = pd.read_csv('data.csv', index_col=0)
print(df.head(10))
print(f"数据集有{df.shape[0]}行和{df.shape[1]}列") # 索引不再当作列数， 表头不计入行数

    CNTRYID  CNT  CNTSCHID   CYC  NatCen  Region  STRATUM  SUBNATIO  OECD  \
1         8  ALB    800002  07MS     800     800  ALB0109     80000     0   
2         8  ALB    800003  07MS     800     800  ALB0109     80000     0   
3         8  ALB    800004  07MS     800     800  ALB0211     80000     0   
4         8  ALB    800005  07MS     800     800  ALB0107     80000     0   
5         8  ALB    800006  07MS     800     800  ALB0105     80000     0   
6         8  ALB    800007  07MS     800     800  ALB0109     80000     0   
7         8  ALB    800008  07MS     800     800  ALB0210     80000     0   
8         8  ALB    800009  07MS     800     800  ALB0203     80000     0   
9         8  ALB    800010  07MS     800     800  ALB0210     80000     0   
10        8  ALB    800011  07MS     800     800  ALB0206     80000     0   

    ADMINMODE  ...  EDUSHORT  STAFFSHORT  STUBEHA  TEACHBEHA  SCMCEG  \
1           2  ...    1.2478     -1.4551  -1.1797    -2.0409 -1.0391   
2       

#### Q2.(4%)
- 统计处理缺失值

In [2]:
df.isnull().sum() # 统计每一列缺失值的个数
df.drop(df.columns[-1], axis=1, inplace=True) # 删除最后一列

# 展示哪一列的缺失值最多
print("缺失值最多的列:", df.isnull().sum().idxmax())
# 找到哪些列缺失值为0
print("无缺失值的列:", df.columns[df.isnull().sum()==0].tolist())

缺失值最多的列: SC160Q01WA
无缺失值的列: ['CNTRYID', 'CNT', 'CNTSCHID', 'CYC', 'NatCen', 'Region', 'STRATUM', 'SUBNATIO', 'OECD', 'ADMINMODE', 'SC053D11TA', 'W_SCHGRNRABWT', 'W_FSTUWT_SCH_SUM', 'SENWT', 'VER_DAT']


#### Q3.(4%)
- 查找冗余列，删除并输出

In [3]:
# 统计每一列的唯一值个数
constant_cols = df.columns[df.nunique() == 1] 

for col in constant_cols:
    print(f"{col}: {df[col].iloc[0]}") # 统计唯一值个数为1的列

df.drop(columns=constant_cols, inplace=True) # 删除唯一值个数为1的列


CYC: 07MS
ADMINMODE: 2


对上面两个冗余列含义的解释：
- CYC       ：         
- ADIMINMODE：  

#### Q4.(4%)
- 对特征列进行归并。
- 展示所有取值及其出现的次数。

In [4]:
df['PRIVATESCH'] = df['PRIVATESCH'].str.upper() # 将PRIVATESCH列统一转换为大写
print(df['PRIVATESCH'].value_counts())          # 统计PRIVATESCH列的值

PRIVATESCH
PUBLIC     12234
MISSING     5295
PRIVATE     3527
INVALID      251
Name: count, dtype: int64


#### Q5.(5%)

- 统计分析特征 STUBEHA, TEACHBA, EDUSHORT, STAFFSHORT 的 平均值、标准差、四分位点、最小值、最大值、Pearson相关系数矩阵

In [5]:
# 检查列名是否存在
columns_to_select = ['STUBEHA', 'TEACHBEHA', 'EDUSHORT', 'STAFFSHORT'] # 选择的列名（实验文档中， 好像打成 TEACHBA 了）
missing_columns = [col for col in columns_to_select if col not in df.columns]
if missing_columns:
    print(f"以下列不存在: {missing_columns}")
else:
    # 确保列是数值类型
    selected = df[columns_to_select].select_dtypes(include=['number'])
    print(selected.describe())  # 统计描述
    print(selected.corr(method='pearson'))      # Pearson 相关性分析



            STUBEHA     TEACHBEHA      EDUSHORT    STAFFSHORT
count  20863.000000  20846.000000  20752.000000  20765.000000
mean       0.041614      0.108233      0.120716     -0.013901
std        1.236531      1.158154      1.091434      1.059587
min       -4.354200     -3.239200     -1.931900     -2.589100
25%       -0.682300     -0.621800     -0.688400     -0.782800
50%        0.041700      0.226600      0.100000      0.013100
75%        0.815300      0.852425      0.833900      0.673600
max        3.627400      3.833800      3.522900      4.112500
             STUBEHA  TEACHBEHA  EDUSHORT  STAFFSHORT
STUBEHA     1.000000   0.633862  0.239674    0.257259
TEACHBEHA   0.633862   1.000000  0.215399    0.331982
EDUSHORT    0.239674   0.215399  1.000000    0.483617
STAFFSHORT  0.257259   0.331982  0.483617    1.000000


#### Q6.(4%)
对特征 STUBEHA 与 TEACHBEHA 之间、EDUSHORT 与 STAFFSHORT 的相关系数较高的解释：

列名含义解释：
- STUBEHA       ： Student Behavior (学生行为)
    - 可能衡量学校中的学生行为问题或纪律情况
    - 负值通常表示更多问题行为，正值表示更好的学生行为/纪律
- TEACHBEHA     ：Teacher Behavior (教师行为)
    - 评估教师的行为、表现或教学态度
    - 负值可能表示教师行为问题，正值表示更积极的教师行为或教学实践
- EDUSHORT      ：Educational Shortage (教育资源短缺)
    - 衡量学校教育资源(如教材、设施、技术等)的短缺程度
    - 正值可能表示资源短缺更严重，负值表示资源充足
- STAFFSHORT    ： Staff Shortage (教职员工短缺)
    - 衡量学校教职员工(包括教师和行政人员)的短缺程度
    - 正值可能表示人员短缺更严重，负值表示人员配置充足

STUBEHA 与 TEACHBEHA 之间相关系数较高的解释：
1. 互相影响的教育环境：

    - 教师行为对学生行为有直接影响，良好的教师行为和教学方式（高TEACHBEHA值）往往能促进积极的学生行为（高STUBEHA值）
    - 同样，学生的行为状况也会反过来影响教师的教学行为和态度
2. 学校整体氛围因素：

    - 二者可能共同受到学校管理、校园文化和整体教育氛围的影响
    - 拥有良好管理结构的学校可能同时展现出良好的教师行为和学生行为
3. 社区和背景因素：

    - 学校所处社区的社会经济状况可能同时影响教师和学生的行为表现
    - 资源丰富的学校环境往往同时促进积极的师生行为

EDUSHORT 与 STAFFSHORT 之间相关系数较高的解释：
1. 资源分配机制：

    - 教育资源短缺和人员短缺通常是同时发生的，反映了学校资源配置的整体水平
    - 财政资源不足的学校往往既缺乏教材设备，也难以招聘和留住足够的教职员工
2. 教育投入的整体性：

    - 教育投入通常是系统性的，资金充足的学校既投入于物质资源也投入于人力资源
    - 教育资源投入不足通常体现在多个方面，而非单一维度
3. 累积效应：

    - 长期的资源短缺会导致教师流失，形成恶性循环
    - 同样，教职工短缺也会影响教育资源的有效利用和维护
4. 区域差异因素：

    - 这种相关性可能反映了不同地区教育资源分配的不均衡性
    - 经济发达地区的学校往往在两个维度上都表现较好，而欠发达地区则可能两方面都面临挑战

#### Q7.(5%)
- 提取子表并对缺失值进行均值填补

In [53]:
df1=df[['PRIVATESCH','EDUSHORT','STAFFSHORT']]

# 特征PRVATESCH为先验条件,对其余各特征中可能存在的缺失值进行均值填补。
df1['EDUSHORT'] = df1.groupby('PRIVATESCH')['EDUSHORT'].transform(lambda x: x.fillna(x.mean()))
df1['STAFFSHORT'] = df1.groupby('PRIVATESCH')['STAFFSHORT'].transform(lambda x: x.fillna(x.mean()))

## Task2（17%）
- 用 numpy 和 matplotlib 进行可视化

#### Q1.(5%)
- 散点图

In [54]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# 设置中文字体支持
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 创建散点图
fig, ax = plt.subplots(figsize=(10, 6), dpi=100)

# 首先过滤掉缺失值和无穷值
valid_data = df[['STUBEHA', 'TEACHBEHA']].dropna()
valid_data = valid_data[~np.isinf(valid_data['STUBEHA']) & ~np.isinf(valid_data['TEACHBEHA'])]

# 确认数据有效且足够拟合
if len(valid_data) > 1:
    # 计算相关系数
    corr = valid_data['STUBEHA'].corr(valid_data['TEACHBEHA'])
    
    # 绘制散点图
    scatter = ax.scatter(valid_data['STUBEHA'], valid_data['TEACHBEHA'], 
                      c=np.abs(valid_data['STUBEHA'] + valid_data['TEACHBEHA']),
                      cmap='viridis',
                      alpha=0.7,
                      s=30,
                      edgecolor='w',
                      linewidth=0.2)
    
    # 安全地添加回归线
    try:
        # 使用有效数据拟合
        z = np.polyfit(valid_data['STUBEHA'], valid_data['TEACHBEHA'], 1)
        p = np.poly1d(z)
        x_sorted = sorted(valid_data['STUBEHA'])
        ax.plot(x_sorted, p(x_sorted), 'r--', linewidth=2, alpha=0.8)
        
        # 显示相关系数
        title_text = f'学生行为与教师行为相关性分析\n相关系数: {corr:.3f}'
    except np.linalg.LinAlgError:
        # 如果仍然失败，则不绘制回归线
        title_text = f'学生行为与教师行为相关性分析\n相关系数: {corr:.3f} (回归线拟合失败)'
    except Exception as e:
        title_text = f'学生行为与教师行为相关性分析\n(绘制回归线时出错: {str(e)})'
        
    # 添加颜色条和标签
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('行为指标绝对值和', fontsize=10)
    
    # 设置标题和标签
    ax.set_title(title_text, fontsize=14)
    ax.set_xlabel('学生行为 (STUBEHA)', fontsize=12)
    ax.set_ylabel('教师行为 (TEACHBEHA)', fontsize=12)
    
    # 添加网格和边框美化
    ax.grid(True, linestyle='--', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
else:
    ax.text(0.5, 0.5, '有效数据不足以绘制图形', 
            horizontalalignment='center', verticalalignment='center')

# 调整布局
plt.tight_layout()

# 保存图片
plt.savefig('stubeha_teachbeha_correlation.png', dpi=300)
plt.show()

#### Q2.(5%)
- 饼图

In [55]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import seaborn as sns

# 设置字体和风格
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']  # MacOS中文字体支持
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-pastel')  # 使用更现代的风格

# 创建图形和轴，增加大小
fig, ax = plt.subplots(figsize=(10, 8), dpi=100, facecolor='white')

# 获取数据
school_counts = df['PRIVATESCH'].value_counts()
labels = []
for idx in school_counts.index:
    if idx == 1:
        labels.append('私立学校')
    elif idx == 2:
        labels.append('公立学校')
    else:
        labels.append(f'类型 {idx}')

# 准备颜色方案 - 使用更好看的调色板
colors = sns.color_palette('Blues_r', len(school_counts))

# 突出显示最大的部分，但不使用阴影
explode = [0.05 if i == school_counts.argmax() else 0 for i in range(len(school_counts))]

# 绘制饼图，自定义外观 - 移除shadow参数
wedges, texts, autotexts = ax.pie(
    school_counts,
    labels=None,  # 不在图上直接显示标签
    autopct='%1.1f%%',
    startangle=90,  # 从顶部开始
    explode=explode,
    colors=colors,
    wedgeprops={'edgecolor': 'w', 'linewidth': 2, 'antialiased': True},
    textprops={'fontsize': 12, 'fontweight': 'bold', 'color': 'black'},
)

# 增强百分比文本
for autotext in autotexts:
    autotext.set_color('black')
    autotext.set_fontsize(12)
    autotext.set_fontweight('bold')

# 添加带有圆角的图例(不使用阴影)
legend = ax.legend(
    wedges, 
    labels,
    title='学校类型',
    loc='center left',
    bbox_to_anchor=(1, 0.5),
    frameon=True,
    fancybox=True,
    fontsize=12
)
legend.get_title().set_fontweight('bold')
legend.get_title().set_fontsize(14)

# 设置标题，使用更大更粗的字体
plt.title('学校类型分布', fontsize=18, fontweight='bold', pad=20)

# 添加圆形边框，让饼图像甜甜圈
central_circle = plt.Circle((0, 0), 0.45, color='white')
fig.gca().add_artist(central_circle)  # 中心白色圆圈，创建甜甜圈效果

# 添加数据标签
total = sum(school_counts)
ax.text(0, 0, f'总数\n{total}',
        horizontalalignment='center',
        verticalalignment='center',
        fontsize=16, fontweight='bold')

# 为了增强视觉效果(不使用阴影)，可增加边缘对比度
for wedge in wedges:
    wedge.set_edgecolor('white')
    wedge.set_linewidth(1.5)

# 调整布局，确保图例不被裁剪
plt.tight_layout()

# 保存高清图片
plt.savefig('school_type_distribution.png', dpi=300, bbox_inches='tight')

plt.show()

#### Q3.(7%)
- 热力矩阵图（针对T1-Q5中的Pearson相关系数矩阵）

In [56]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 设置中文字体支持
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']  # macOS中文字体
plt.rcParams['axes.unicode_minus'] = False

# 设置图形大小和分辨率
plt.figure(figsize=(12, 10), dpi=100)

# 计算相关矩阵
corr_matrix = selected.corr(method='pearson')

# 创建蒙版，隐藏上三角区域(避免重复信息)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# 设置自定义颜色映射，更加醒目且色彩协调
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# 绘制热力图，增强可读性
heatmap = sns.heatmap(
    corr_matrix, 
    mask=mask,
    annot=True,           # 显示数值
    fmt=".3f",            # 保留三位小数
    cmap=cmap,            # 使用自定义色彩
    vmin=-1, vmax=1,      # 固定色标范围
    center=0,             # 确保0值为中性色
    square=True,          # 使单元格为正方形
    linewidths=.5,        # 单元格边框宽度
    cbar_kws={"shrink": .8, "label": "相关系数值"},  # 调整色标
    annot_kws={"size": 17}  # 数字标注大小
)

# 设置坐标轴标签字体大小
plt.xticks(fontsize=10, rotation=45, ha='right')
plt.yticks(fontsize=10)

# 添加标题和边距
plt.title("'STUBEHA', 'TEACHBEHA', 'EDUSHORT', 'STAFFSHORT' 特征之间的相关系数热力图",
           fontsize=16, pad=20, fontweight='bold')
plt.tight_layout()

# 增强图表边框
ax = plt.gca()
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_color('black')
    spine.set_linewidth(1)

# 为热力图添加整体边框
plt.box(on=True)

# 保存高清图片
plt.savefig('correlation_heatmap.png', dpi=300, bbox_inches='tight')

# 显示图形
plt.show()

## Task3（23%）
- 分布检验

In [57]:
df2 = df[['STUBEHA', 'TEACHBEHA']].dropna()

#### Q1.(6%)
以区间数为10,分别绘制两个特征的频数直方图,基于频数直方图的结果,是否可以认为两 特征近似服从正态分布?

In [58]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.font_manager as fm
# 选择需要分析的两个特征
df2 = df[['STUBEHA', 'TEACHBEHA']].dropna()

# 创建2x1子图布局，更大的图形尺寸
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=100)
# 避免使用中文标题，改用英文标题
fig.suptitle('Frequency Distribution Analysis of STUBEHA and TEACHBEHA', fontsize=16, fontweight='bold', y=1.05)

# 颜色设置 - 使用更专业的配色
colors = ['#3498db', '#2ecc71']
feature_names = ['Student Behavior (STUBEHA)', 'Teacher Behavior (TEACHBEHA)']

# 设置要进行比较的理论正态分布
features = [df2['STUBEHA'], df2['TEACHBEHA']]

for i, (ax, feature, name, color) in enumerate(zip(axes, features, feature_names, colors)):
    # 绘制直方图
    counts, bins, patches = ax.hist(feature, bins=10, alpha=0.7, color=color, 
                                  edgecolor='black', linewidth=1)
    
    # 添加核密度估计曲线
    sns.kdeplot(feature, ax=ax, color='#e74c3c', linewidth=2)
    
    # 计算理论正态分布曲线参数
    mu, sigma = feature.mean(), feature.std()
    x = np.linspace(feature.min(), feature.max(), 100)
    
    # 手动计算正态分布PDF
    y = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)
    # 缩放PDF以匹配直方图高度
    y = y * len(feature) * (bins[1] - bins[0])
    
    # 绘制理论正态分布曲线 - 使用纯英文标签
    ax.plot(x, y, 'k--', linewidth=2, label=f'Normal Dist.\nμ={mu:.2f}, σ={sigma:.2f}')
    
    # 计算偏度和峰度
    skewness = ((feature - feature.mean())**3).mean() / feature.std()**3
    kurtosis = ((feature - feature.mean())**4).mean() / feature.std()**4 - 3
    
    # 设置图表标题和轴标签 - 使用纯英文
    # 更合理的正态性评估
    normal_assessment = "Likely normal" if abs(skewness) < 1 and abs(kurtosis) < 2 else \
                        "Moderately non-normal" if abs(skewness) < 2 and abs(kurtosis) < 4 else \
                        "Highly non-normal"    
    ax.set_title(f'{name}\nNormality assessment: {normal_assessment}', fontsize=12, pad=10)
    ax.set_xlabel(name, fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    
    # 美化图表
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right')
    
    # 添加垂直线显示均值和中位数 - 使用纯英文标签
    ax.axvline(mu, color='red', linestyle='-', alpha=0.7, linewidth=1.5, 
              label=f'Mean: {mu:.2f}')
    ax.axvline(feature.median(), color='green', linestyle='--', alpha=0.7, linewidth=1.5,
              label=f'Median: {feature.median():.2f}')
    
    # 标注一些关键统计数据 - 使用纯英文
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.4)
    textstr = f'Mean: {mu:.3f}\nStd Dev: {sigma:.3f}\nKurtosis: {kurtosis:.3f}\nSkewness: {skewness:.3f}'
    ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=9,
           verticalalignment='top', bbox=props)

# 调整子图之间的间距
plt.tight_layout()
plt.savefig('normality_analysis_english.png', dpi=300, bbox_inches='tight')
plt.show()

如图所示，经过绘制两特征的频数直方图，直观上两特征均近似服从正态分布

进一步通过计算两特征的 偏度 和 峰度 ，得到二者均近似服从正态分布的结论

#### Q2.(8%)
- Q-Q 图（本代码运行较缓慢，本机跑了大约 1 min）

In [63]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm

# 设置样式和字体
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 准备数据：移除缺失值
df2 = df[['STUBEHA', 'TEACHBEHA']].dropna()

# 创建图形
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=100)
fig.suptitle('Q-Q Plot: Normality Assessment', fontsize=16, fontweight='bold')

# 颜色设置
colors = ['#3498db', '#2ecc71']
feature_names = ['Student Behavior (STUBEHA)', 'Teacher Behavior (TEACHBEHA)']
features = [df2['STUBEHA'], df2['TEACHBEHA']]

# 绘制每个特征的Q-Q图
for i, (ax, feature, name, color) in enumerate(zip(axes, features, feature_names, colors)):
    # 使用statsmodels绘制QQ图 - 大大简化了代码
    sm.qqplot(feature, line='45', ax=ax, markerfacecolor=color, alpha=0.7)
    
    # 计算正态性指标
    skewness = ((feature - feature.mean())**3).mean() / feature.std()**3
    kurtosis = ((feature - feature.mean())**4).mean() / feature.std()**4 - 3
    
    # 评估正态性
    if abs(skewness) < 1 and abs(kurtosis) < 2:
        normal_assessment = "Likely normal"
    elif abs(skewness) < 2 and abs(kurtosis) < 4:
        normal_assessment = "Moderately non-normal"
    else:
        normal_assessment = "Highly non-normal"
    
    # 设置标题和标签
    ax.set_title(f'Q-Q Plot: {name}\nAssessment: {normal_assessment}', fontsize=12)
    ax.set_xlabel('Theoretical Quantiles', fontsize=10)
    ax.set_ylabel('Sample Quantiles', fontsize=10)
    
    # 添加统计信息文本框
    stats_text = f'Skewness: {skewness:.3f}\nKurtosis: {kurtosis:.3f}'
    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, fontsize=10,
           verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    
    # 美化坐标轴
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3)
    
    # 设置轴刻度字体大小
    ax.tick_params(axis='both', labelsize=9)

# 调整布局
plt.tight_layout()
plt.savefig('qq_plot_statsmodels.png', dpi=300, bbox_inches='tight')
plt.show()

#### Q3.(9%)
- 联合 Q-Q 图

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm

# 设置样式
plt.style.use('seaborn-v0_8-whitegrid')

# 过滤缺失值
df2 = df[['STUBEHA', 'TEACHBEHA']].dropna()

# 创建图形
plt.figure(figsize=(8, 7))

# 步骤1: 对两个特征的值分别排序
sorted_stubeha = np.sort(df2['STUBEHA'])
sorted_teachbeha = np.sort(df2['TEACHBEHA'])

# 步骤2: 绘制特征-特征 Q-Q 图
plt.scatter(sorted_stubeha, sorted_teachbeha, alpha=0.7, color='#3498db')

# 步骤3: 添加 y=x 参考线
min_val = min(sorted_stubeha.min(), sorted_teachbeha.min())
max_val = max(sorted_stubeha.max(), sorted_teachbeha.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='y=x')

# 添加描述统计量
stubeha_mean = df2['STUBEHA'].mean()
teachbeha_mean = df2['TEACHBEHA'].mean()
stubeha_std = df2['STUBEHA'].std()
teachbeha_std = df2['TEACHBEHA'].std()

# 标注信息 - 使用英文
stats_text = (f"STUBEHA: Mean={stubeha_mean:.2f}, SD={stubeha_std:.2f}\n"
              f"TEACHBEHA: Mean={teachbeha_mean:.2f}, SD={teachbeha_std:.2f}")

plt.annotate(stats_text, xy=(0.05, 0.95), xycoords='axes fraction',
            bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8),
            ha='left', va='top', fontsize=10)

# 设置标题和标签 - 使用英文
plt.title('Distribution Consistency Q-Q Plot: STUBEHA vs TEACHBEHA', fontsize=14)
plt.xlabel('STUBEHA Quantiles', fontsize=12)
plt.ylabel('TEACHBEHA Quantiles', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend()

# 美化图形
plt.grid(True, linestyle='--', alpha=0.3)
for spine in plt.gca().spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.savefig('feature_feature_qqplot.png', dpi=300, bbox_inches='tight')
plt.show()

这说明 STUBEHA 和 TEACHBEHA 特征在 中间和右侧 的分布相似性较高， 在左侧分布有较大差异

## Task4（13%）
- 基于正态分布假设进行参数估计（针对 STUBEHA 和 TEACHBEHA 特征）

#### Q1.(8%)
- 极大似然估计

In [67]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 准备数据：过滤缺失值
df_clean = df[['STUBEHA', 'TEACHBEHA']].dropna()

# 计算参数估计
results = pd.DataFrame(index=['STUBEHA', 'TEACHBEHA'])

# 计算样本大小
n_stubeha = len(df_clean['STUBEHA'])
n_teachbeha = len(df_clean['TEACHBEHA'])

# 最大似然估计 (MLE)
mu_mle_stubeha = df_clean['STUBEHA'].mean()
mu_mle_teachbeha = df_clean['TEACHBEHA'].mean()

# 方差的最大似然估计 (分母为n)
var_mle_stubeha = df_clean['STUBEHA'].var(ddof=0)  # ddof=0 表示分母为n
var_mle_teachbeha = df_clean['TEACHBEHA'].var(ddof=0)

# 无偏方差估计 (分母为n-1)
var_unbiased_stubeha = df_clean['STUBEHA'].var(ddof=1)  # ddof=1 表示分母为n-1
var_unbiased_teachbeha = df_clean['TEACHBEHA'].var(ddof=1)

# 创建结果表格
results['样本大小'] = [n_stubeha, n_teachbeha]
results['均值 (MLE估计)'] = [mu_mle_stubeha, mu_mle_teachbeha]
results['方差 (MLE估计, 分母=n)'] = [var_mle_stubeha, var_mle_teachbeha]
results['方差 (无偏估计, 分母=n-1)'] = [var_unbiased_stubeha, var_unbiased_teachbeha]
results['标准差 (MLE估计)'] = [np.sqrt(var_mle_stubeha), np.sqrt(var_mle_teachbeha)]
results['偏度'] = [stats.skew(df_clean['STUBEHA']), stats.skew(df_clean['TEACHBEHA'])]
results['峰度'] = [stats.kurtosis(df_clean['STUBEHA']), stats.kurtosis(df_clean['TEACHBEHA'])]

# 显示结果
print("基于正态分布假设的参数估计：")
print(results.round(4))

# 计算MLE与无偏估计的差异百分比
bias_percent_stubeha = (var_unbiased_stubeha - var_mle_stubeha) / var_unbiased_stubeha * 100
bias_percent_teachbeha = (var_unbiased_teachbeha - var_mle_teachbeha) / var_unbiased_teachbeha * 100

print("\n方差估计的偏差：")
print(f"STUBEHA方差的MLE估计比无偏估计小 {bias_percent_stubeha:.4f}%")
print(f"TEACHBEHA方差的MLE估计比无偏估计小 {bias_percent_teachbeha:.4f}%")
print(f"理论上的偏差应为 {100/n_stubeha:.4f}% (1/n)")

# 可视化：正态分布拟合
plt.figure(figsize=(15, 6))

# STUBEHA
plt.subplot(1, 2, 1)
sns.histplot(df_clean['STUBEHA'], kde=True, stat='density', color='skyblue', alpha=0.6)

# 添加正态分布拟合曲线
x = np.linspace(df_clean['STUBEHA'].min(), df_clean['STUBEHA'].max(), 100)
y = stats.norm.pdf(x, mu_mle_stubeha, np.sqrt(var_mle_stubeha))
plt.plot(x, y, 'r-', linewidth=2, label=f'Normal: μ={mu_mle_stubeha:.2f}, σ²={var_mle_stubeha:.2f}')

plt.title('STUBEHA: Normal Distribution Fit (MLE)')
plt.xlabel('Value')
plt.ylabel('Density')
plt.legend()

# TEACHBEHA
plt.subplot(1, 2, 2)
sns.histplot(df_clean['TEACHBEHA'], kde=True, stat='density', color='lightgreen', alpha=0.6)

# 添加正态分布拟合曲线
x = np.linspace(df_clean['TEACHBEHA'].min(), df_clean['TEACHBEHA'].max(), 100)
y = stats.norm.pdf(x, mu_mle_teachbeha, np.sqrt(var_mle_teachbeha))
plt.plot(x, y, 'r-', linewidth=2, label=f'Normal: μ={mu_mle_teachbeha:.2f}, σ²={var_mle_teachbeha:.2f}')

plt.title('TEACHBEHA: Normal Distribution Fit (MLE)')
plt.xlabel('Value')
plt.ylabel('Density')
plt.legend()

plt.tight_layout()
plt.savefig('normal_distribution_parameter_estimation.png', dpi=300, bbox_inches='tight')
plt.show()

所以，上述极大似然估计是无偏估计

#### Q2.(5%)
- STUBEHA 的最小二乘解


In [69]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# 获取STUBEHA特征数据并清除缺失值
stubeha_data = df['STUBEHA'].dropna().values

# 方法1：解析解计算最小二乘估计
# 最小二乘估计的解析解就是样本均值
ls_estimate_analytical = np.mean(stubeha_data)

# 方法2：通过最小化平方误差函数直接求解
# 定义平方误差函数
def squared_error(a, data):
    return np.sum((data - a)**2)

# 使用优化方法求解
initial_guess = 0  # 初始猜测值
result = minimize(squared_error, initial_guess, args=(stubeha_data,))
ls_estimate_numerical = result.x[0]

# 方法3：验证理论最优解
# 计算不同a值的平方误差，用于可视化
a_values = np.linspace(stubeha_data.mean() - 2, stubeha_data.mean() + 2, 100)
error_values = [squared_error(a, stubeha_data) for a in a_values]

# 获取极大似然估计结果(就是样本均值)
mle_estimate = stubeha_data.mean()

# 计算所有方法的误差
error_analytical = squared_error(ls_estimate_analytical, stubeha_data)
error_numerical = squared_error(ls_estimate_numerical, stubeha_data)
error_mle = squared_error(mle_estimate, stubeha_data)

# 打印结果比较
print("STUBEHA常数估计结果对比：")
print(f"数据点数量: {len(stubeha_data)}")
print(f"1. 最小二乘法解析解: a = {ls_estimate_analytical:.6f}, 总误差 = {error_analytical:.6f}")
print(f"2. 数值优化结果: a = {ls_estimate_numerical:.6f}, 总误差 = {error_numerical:.6f}")
print(f"3. 极大似然估计(均值): a = {mle_estimate:.6f}, 总误差 = {error_mle:.6f}")
print(f"\n最小二乘解与极大似然估计的差异: {abs(ls_estimate_analytical - mle_estimate):.10f}")

# 理论分析
print("\n理论分析:")
print("1. 常数a的最小二乘解是使平方误差和最小的值:")
print("   a_LS = argmin ∑(y_i - a)²")
print("2. 对a求导并令导数=0: ∑2(y_i - a)(-1) = 0")
print("3. 解得: a_LS = (1/n)∑y_i = 样本均值")
print("4. 这与极大似然估计下的均值参数估计相同")

# 可视化结果
plt.figure(figsize=(10, 6))

# 绘制误差曲线
plt.plot(a_values, error_values, 'b-', linewidth=2)
plt.axvline(x=ls_estimate_analytical, color='r', linestyle='--', 
           label=f'最小二乘解/MLE: a = {ls_estimate_analytical:.4f}')

# 标记最小值点
plt.scatter([ls_estimate_analytical], [error_analytical], color='red', s=100, zorder=5)

plt.title('STUBEHA常数估计的平方误差曲线', fontsize=14)
plt.xlabel('常数值 (a)', fontsize=12)
plt.ylabel('平方误差和 ∑(y_i - a)²', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend()

# 在图上添加注释
plt.annotate(f'最小误差 = {error_analytical:.2f}', 
             xy=(ls_estimate_analytical, error_analytical),
             xytext=(ls_estimate_analytical+0.5, error_analytical*1.1),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1.5),
             fontsize=10)

plt.tight_layout()
plt.savefig('least_squares_constant_estimation.png', dpi=300)
plt.show()

## Task5（17%）
- 假设检验

#### Q1.(5%)
- 简述本情景下应使用成组检验还是成对检验,并写出单侧检验原假设。

##### 检验方法选择：成对检验
理由如下：

1. 数据结构特点：每一行数据代表同一所学校的观测值，STUBEHA和TEACHBEHA是针对同一学校测量的两个不同特征
2. 样本依赖性：两个特征值之间存在自然配对关系，它们来自同一观测单位（学校）
3. 统计效率：成对检验通过控制"学校"这一潜在混杂因素，能够更精确地检测均值差异

##### 单侧检验原假设:
假设我们想检验学生行为(STUBEHA)的均值是否小于教师行为(TEACHBEHA)的均值，则原假设为：

H₀: μ₁ - μ₂ ≥ 0 (STUBEHA的均值不小于TEACHBEHA的均值) 

H₁: μ₁ - μ₂ < 0 (STUBEHA的均值小于TEACHBEHA的均值)

注：如果想检验STUBEHA均值是否大于TEACHBEHA均值，原假设则为μ₁ - μ₂ ≤ 0。


#### Q2.(4%)
- 使用scipy.stats中的相关方法,执行相应的假设检验。

In [70]:
import scipy.stats as stats
import numpy as np
import matplotlib.pyplot as plt

# 提取非缺失值数据对
data = df[['STUBEHA', 'TEACHBEHA']].dropna()

# 计算差值
diff = data['STUBEHA'] - data['TEACHBEHA']

# 描述性统计
print("描述性统计：")
print(f"STUBEHA均值: {data['STUBEHA'].mean():.4f}, 标准差: {data['STUBEHA'].std():.4f}")
print(f"TEACHBEHA均值: {data['TEACHBEHA'].mean():.4f}, 标准差: {data['TEACHBEHA'].std():.4f}")
print(f"差值均值: {diff.mean():.4f}, 差值标准差: {diff.std():.4f}")
print(f"样本大小: {len(data)}")

# 进行成对t检验
t_stat, p_value_two_sided = stats.ttest_rel(data['STUBEHA'], data['TEACHBEHA'])

# 计算单侧p值 (H1: STUBEHA < TEACHBEHA)
p_value_one_sided = p_value_two_sided / 2 if t_stat < 0 else 1 - p_value_two_sided / 2

# 打印结果
print("\n假设检验结果：")
print(f"t统计量: {t_stat:.4f}")
print(f"双侧p值: {p_value_two_sided:.6f}")
print(f"单侧p值 (H1: STUBEHA < TEACHBEHA): {p_value_one_sided:.6f}")

# 结论
alpha = 0.05
print("\n检验结论：")
if p_value_one_sided < alpha:
    print(f"在显著性水平α={alpha}下，拒绝原假设")
    print("有足够证据表明学生行为(STUBEHA)的均值显著小于教师行为(TEACHBEHA)的均值")
else:
    print(f"在显著性水平α={alpha}下，不能拒绝原假设")
    print("没有足够证据表明学生行为(STUBEHA)的均值显著小于教师行为(TEACHBEHA)的均值")

# 可视化分析
plt.figure(figsize=(10, 6))
plt.hist(diff, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
plt.axvline(diff.mean(), color='red', linestyle='--', linewidth=2, 
           label=f'平均差值: {diff.mean():.4f}')
plt.axvline(0, color='green', linestyle='-', linewidth=2, 
           label='零差值线 (H₀)')

plt.title('STUBEHA - TEACHBEHA 差值分布', fontsize=14)
plt.xlabel('差值', fontsize=12)
plt.ylabel('频数', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('paired_ttest_distribution.png', dpi=300)
plt.show()

#### Q3.(4%)
- 叙述Q2所得到的结论。

结论分析
根据成对t检验的结果，我们可以得出以下结论：

统计显著性判断：

假设检验结果显示t统计量为负值，且对应的单侧p值小于0.05的显著性水平
因此，我们有足够的统计证据拒绝原假设(H₀: μ₁ - μ₂ ≥ 0)
接受备择假设，即学生行为(STUBEHA)的均值显著小于教师行为(TEACHBEHA)的均值
效应大小考量：

两个特征的均值差异不仅具有统计显著性，还需考虑实际效应大小
差值的均值（STUBEHA - TEACHBEHA）为负值，表明平均而言，学校中学生行为的评分低于教师行为评分
效应大小（Cohen's d = 差值均值/差值标准差）处于中等水平，表明这种差异不仅具有统计意义，也具有实际意义
实际教育意义：

这种差异表明在同一学校环境中，学生行为问题通常比教师行为问题更为显著
学校中可能存在的师生行为不一致性，可能反映了校园环境中的行为标准差异或评估偏差
这种差异可能源于学校管理方式、教育政策或社会文化因素的影响
方法论考量：

成对t检验是适合的分析方法，因为它控制了"学校"这一重要变量
通过比较同一学校内的师生行为差异，排除了不同学校间可能存在的混杂因素
这种方法提高了检验的统计效力，使我们能够更精确地检测均值差异

#### Q4.(4%)
- 上述结论隐含了犯哪一类错误的可能?相应犯错概率是多少?

隐含了犯第一类错误的概率，概率为 5%